In [1]:
from ultralytics import YOLO
import torch

# 1. GPU 메모리 캐시 비우기 (안전한 학습 시작을 위해)
torch.cuda.empty_cache()

# 2. YOLOv10-S (Small) 사전학습 모델 로드
# v10 모델을 사용하려면 ultralytics 패키지가 최신 버전이어야 합니다.
model = YOLO('yolov10s.pt') 

# 3. 모델 학습 시작 (Fine-tuning)
if __name__ == '__main__':
    results = model.train(
        data='yolo/dataset.yaml', # 앞서 전처리 단계에서 만든 설정 파일 경로
        epochs=50,                # 전체 데이터를 반복 학습할 횟수 (시간 여유에 따라 50~100 권장)
        imgsz=640,                # 입력 이미지 크기 (640이 표준)
        batch=16,                 # 5060Ti VRAM(8GB/16GB) 환경에서 안정적인 배치 사이즈
        device=0,                 # 0번 GPU 사용
        project='VQA_YOLO',       # 결과물이 저장될 최상위 폴더 이름
        name='yolov10_run_1',     # 이번 학습 세션의 이름
        patience=15,              # 15 epoch 동안 성능 향상이 없으면 조기 종료 (과적합 방지)
        save=True,                # 학습된 가중치 저장 활성화
        exist_ok=True             # 동일한 세션 이름이 있으면 덮어쓰기 허용
    )
    
    print("\n🎉 YOLO 학습이 완료되었습니다!")
    print("가장 성능이 좋은 모델 가중치는 'VQA_YOLO/yolov10_run_1/weights/best.pt' 에 저장되었습니다.")

Ultralytics 8.4.33 🚀 Python-3.11.12 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov10_run_1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, persp

In [2]:
## [4] Qwen 파인튜닝을 실행하기 위한 메모리 청소

import gc
import torch

# 1. YOLO 모델 변수를 메모리에서 강제 삭제
del model 
del results

# 2. 파이썬 가비지 컬렉터 강제 실행 (찌꺼기 청소)
gc.collect()

# 3. 파이토치가 쥐고 있는 GPU 캐시 메모리 강제 반환 (가장 중요!)
torch.cuda.empty_cache()

print("🧹 생선을 썰던 도마를 완벽하게 닦아냈습니다. VRAM이 초기화되었습니다!")

🧹 생선을 썰던 도마를 완벽하게 닦아냈습니다. VRAM이 초기화되었습니다!


In [3]:
## [4] Qwen 파인튜닝

!pip install transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

In [5]:
import torch
from datasets import load_dataset
from transformers import (
    AutoProcessor, 
    Qwen2VLForConditionalGeneration, 
    BitsAndBytesConfig, 
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments # (삭제해도 됨)
from trl import SFTTrainer, SFTConfig  # SFTConfig 추가!

# ==========================================
# 1. 초기화 및 GPU 메모리 정리
# ==========================================
torch.cuda.empty_cache()
model_id = "Qwen/Qwen2-VL-2B-Instruct"

print("⏳ Qwen2-VL 모델과 프로세서를 로드하는 중입니다...")

# ==========================================
# 2. 5060Ti 맞춤형 4-bit 양자화 (QLoRA) 설정
# ==========================================
# 모델을 4-bit로 압축하여 로드 (VRAM 사용량을 극적으로 줄여줌)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 모델 로드 (학습용)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 프로세서 로드 (텍스트와 이미지를 모델이 이해할 수 있게 변환하는 도구)
processor = AutoProcessor.from_pretrained(model_id)

# k-bit 학습을 위한 모델 준비
model = prepare_model_for_kbit_training(model)

# ==========================================
# 3. LoRA 어댑터 설정 (뇌의 일부분만 학습)
# ==========================================
# 전체 모델을 다 학습하면 5060Ti가 터집니다. 
# LoRA를 사용해 핵심 주의집중(Attention) 모듈만 콕 집어서 가볍게 학습시킵니다.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"], # 핵심 모듈만 타겟팅
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters() # 학습할 파라미터 비율 출력 (보통 1~2% 내외)

# ==========================================
# 4. 데이터셋 로드 (미리 만들어둔 1회성 정답지)
# ==========================================
# Phase 2에서 만들어둔 JSONL 파일을 불러옵니다.
dataset = load_dataset('json', data_files={'train': 'train_vlm_resized.jsonl'})

# ==========================================
# 5. SFTTrainer 학습 설정 및 시작
# ==========================================
training_args = SFTConfig(          # 💡 TrainingArguments를 SFTConfig로 변경
    output_dir="VQA_Qwen_LoRA",       
    num_train_epochs=3,               
    per_device_train_batch_size=2,    
    gradient_accumulation_steps=4,    
    optim="paged_adamw_8bit",         
    save_strategy="epoch",            
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,                        
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    dataset_text_field="messages",    # 💡 SFTTrainer에 있던 것을 이쪽으로 옮김!
)

# 트레이너 객체 생성
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    peft_config=lora_config,
    # 💡 여기서 dataset_text_field="messages" 는 지워줍니다!
)

print("\n🔥 Qwen-VL 학습을 시작합니다! (GPU 팬이 강하게 돌 수 있습니다)")

if __name__ == '__main__':
    # 학습 시작
    trainer.train()
    
    # 학습이 끝난 후 최종 LoRA 가중치를 안전하게 디스크에 저장
    trainer.model.save_pretrained("VQA_Qwen_LoRA/final_model")
    processor.save_pretrained("VQA_Qwen_LoRA/final_model")
    print("\n🎉 VLM 학습이 완료되었습니다! 최종 모델이 'VQA_Qwen_LoRA/final_model'에 저장되었습니다.")

⏳ Qwen2-VL 모델과 프로세서를 로드하는 중입니다...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 2,211,164,672 || trainable%: 0.0985


/opt/conda/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/3326 [00:00<?, ? examples/s]

...


🔥 Qwen-VL 학습을 시작합니다! (GPU 팬이 강하게 돌 수 있습니다)


/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.618800
20,2.339600
30,1.812900
40,1.144900
50,0.839500
60,0.767800
70,0.783400
80,0.648200
90,0.619300
100,0.508300


/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/conda/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



🎉 VLM 학습이 완료되었습니다! 최종 모델이 'VQA_Qwen_LoRA/final_model'에 저장되었습니다.
